<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


This exercise will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [2]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [3]:
import pandas as pd

# Assuming df is your DataFrame
# 1. Display column data types and non-null counts
print("DataFrame info:")
df.info()

# 2. Display number of missing values per column
print("\nMissing values per column:")
missing_values = df.isna().sum()
print(missing_values)

# 3. Optional: Create a summary DataFrame
summary = pd.DataFrame({
    'DataType': df.dtypes,
    'NonMissingCount': df.notna().sum(),
    'MissingCount': df.isna().sum()
})

print("\nSummary of dataset:")
print(summary)

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Columns: 114 entries, ResponseId to JobSat
dtypes: float64(13), int64(1), str(100)
memory usage: 195.7 MB

Missing values per column:
ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10631
                       ...  
JobSatPoints_11        35992
SurveyLength            9255
SurveyEase              9199
ConvertedCompYearly    42002
JobSat                 36311
Length: 114, dtype: int64

Summary of dataset:
                    DataType  NonMissingCount  MissingCount
ResponseId             int64            65437             0
MainBranch               str            65437             0
Age                      str            65437             0
Employment               str            65437             0
RemoteWork               str            54806         10631
...                      ...              ...           ...

<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [4]:
# Generate basic statistics for all numerical columns
numerical_stats = df.describe()

print("Basic statistics for numerical columns:")
print(numerical_stats)

Basic statistics for numerical columns:
         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000  

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [5]:
# Display all unique countries
unique_countries = df['Country'].unique()
print("Unique Countries:",unique_countries)
# Count how many times each country appears
country_counts = df['Country'].value_counts(dropna=False)
print("Country counts:", country_counts)
missing_countries = df['Country'].isna().sum()
print("Missing Country entries:", missing_countries)
valid_countries = ["United States of America", "India", "Canada", "United Kingdom", "Germany"]  # Example
invalid_entries = df[~df['Country'].isin(valid_countries)]['Country'].unique()
print("Potential inconsistent entries:", invalid_entries)

Unique Countries: <ArrowStringArray>
[                            'United States of America',
 'United Kingdom of Great Britain and Northern Ireland',
                                               'Canada',
                                               'Norway',
                                           'Uzbekistan',
                                               'Serbia',
                                               'Poland',
                                          'Philippines',
                                             'Bulgaria',
                                          'Switzerland',
 ...
                                'Saint Kitts and Nevis',
                                               'Monaco',
                   'Micronesia, Federated States of...',
                                                'Haiti',
                                                    nan,
                                                'Nauru',
                                              

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [6]:
country_mapping = {
    "United States of America": "United States",
    "USA": "United States",
    "U.S.A.": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "England": "United Kingdom",
    "Deutschland": "Germany",
    # Add more mappings as needed
}
edlevel_mapping = {
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": "Bachelor's degree",
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's degree",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional degree",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary School",
    "Associate degree (A.A., A.S., etc.)": "Associate"
    # Add more mappings as needed
}
# Standardize Country
df['Country'] = df['Country'].str.strip()
df['Country'] = df['Country'].replace(country_mapping)

# Standardize EdLevel
df['EdLevel'] = df['EdLevel'].str.strip()
df['EdLevel'] = df['EdLevel'].replace(edlevel_mapping)

# Check unique values after standardization
print(df['Country'].unique())
print(df['EdLevel'].unique())


<StringArray>
[                                       'United States',
 'United Kingdom of Great Britain and Northern Ireland',
                                               'Canada',
                                               'Norway',
                                           'Uzbekistan',
                                               'Serbia',
                                               'Poland',
                                          'Philippines',
                                             'Bulgaria',
                                          'Switzerland',
 ...
                                'Saint Kitts and Nevis',
                                               'Monaco',
                   'Micronesia, Federated States of...',
                                                'Haiti',
                                                    nan,
                                                'Nauru',
                                              'Liberia',
            

### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [6]:
import pandas as pd

# One-hot encode the Employment column
employment_dummies = pd.get_dummies(df['Employment'], prefix='Employment')

# Add the new columns to the original DataFrame
df = pd.concat([df, employment_dummies], axis=1)

# Optional: Drop the original Employment column if no longer needed
# df = df.drop('Employment', axis=1)

# Display the first few rows to check
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

In [40]:
df.head()

,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,"Employment_Student, full-time;Not employed, but looking for work;Not employed, and not looking for work;Student, part-time","Employment_Student, full-time;Not employed, but looking for work;Retired","Employment_Student, full-time;Not employed, but looking for work;Student, part-time","Employment_Student, full-time;Retired","Employment_Student, full-time;Student, part-time","Employment_Student, full-time;Student, part-time;Employed, part-time","Employment_Student, full-time;Student, part-time;Retired","Employment_Student, part-time","Employment_Student, part-time;Employed, part-time","Employment_Student, part-time;Retired"
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,False,False,False,False,False,False,False,False,False,False
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,Bachelor's degree,Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,False,False,False,False,False,False,False,False,False,False
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,Master's degree,Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,False,False,False,False,False,False,False,False,False,False
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,False,False,False,False,False,False,False,False,False,False
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,Secondary School,"Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,False,False,False,False,False,False,False,False,False,False


### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [7]:
# Count missing values per column
missing_counts = df.isna().sum()

# Sort columns by number of missing values (descending)
missing_counts_sorted = missing_counts.sort_values(ascending=False)

# Display columns with the most missing values
print("Columns with the highest number of missing values:")
print(missing_counts_sorted)
print(missing_counts_sorted.head(10))

Columns with the highest number of missing values:
AINextMuch less integrated                                              64289
AINextLess integrated                                                   63082
AINextNo change                                                         52939
AINextMuch more integrated                                              51999
EmbeddedAdmired                                                         48704
                                                                        ...  
Employment_Student, full-time;Student, part-time;Employed, part-time        0
Employment_Student, full-time;Student, part-time;Retired                    0
Employment_Student, part-time                                               0
Employment_Student, part-time;Employed, part-time                           0
Employment_Student, part-time;Retired                                       0
Length: 224, dtype: int64
AINextMuch less integrated       64289
AINextLess integrated     

<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [10]:
# Fill missing values with the mean
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].mean())

# Verify
print("Missing values after mean imputation:", df['ConvertedCompYearly'].isna().sum())
# Fill missing values with the median
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].median())

# Verify
print("Missing values after median imputation:", df['ConvertedCompYearly'].isna().sum())
numerical_cols = ['ConvertedCompYearly', 'WorkExp']
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

Missing values after mean imputation: 0
Missing values after median imputation: 0

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [11]:
# Fill missing values with the most frequent value
df['RemoteWork'] = df['RemoteWork'].fillna(df['RemoteWork'].mode()[0])

# Verify
print("Missing values after imputation:", df['RemoteWork'].isna().sum())
categorical_cols = ['RemoteWork', 'Employment', 'EdLevel']  # add other categorical columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

Missing values after imputation: 0

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [14]:
!pip install -U scikit-learn
from sklearn.preprocessing import MinMaxScaler

# Min-Max normalization using Pandas
df['ConvertedCompYearly_MinMax'] = (
    (df['ConvertedCompYearly'] - df['ConvertedCompYearly'].min()) /
    (df['ConvertedCompYearly'].max() - df['ConvertedCompYearly'].min())
)

# Verify
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())


scaler = MinMaxScaler()
df['ConvertedCompYearly_MinMax'] = scaler.fit_transform(df[['ConvertedCompYearly']])

# Verify
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 133.2 MB/s eta 0:00:0000:01
   ConvertedCompYearly  ConvertedCompYearly_MinMax
0         86155.287263                      0.0053
1         86155.287263                      0.0053
2         86155.287263                      0.0053
3         86155.287263                      0.0053
4         86155.287263                      0.0053
   ConvertedCompYearly  ConvertedCompYearly_MinMax
0         86155.287263                      0.0053
1         86155.287263                      0.0053
2         86155.287263                      0.0053
3         86155.287263                      0.0053
4         86155.287263                      0.0053


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [ ]:
## Write youhttps://jupyterlabnext-3-labs-prod-jupyterlab-us-east-0.labs.cognitiveclass.ai/user/joannehatche/lab/tree/DA0321EN/labs/module%202/M2DataWrangling-lab-v2.ipynb#6.2-Log-transform-the-ConvertedCompYearly-column-to-reduce-skewness.r code here

In [21]:
df["YearsCodePro"]

0        NaN
1         17
2         27
3        NaN
4        NaN
        ... 
65432      3
65433    NaN
65434      5
65435      2
65436    NaN
Name: YearsCodePro, Length: 65437, dtype: str

### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [21]:
# Function to categorize experience
def categorize_experience(years):
    try:
        years = float(years)  # Convert to float if it's a string
    except:
        return "Unknown"  # Handle missing or invalid values

    if years < 2:
        return "Junior"
    elif 2 <= years < 5:
        return "Mid-level"
    elif 5 <= years < 10:
        return "Senior"
    else:
        return "Expert"
        
df['WorkExp'] = df['YearsCodePro'].apply(categorize_experience)
# VerifyExp
print(df[['YearsCodePro', 'WorkExp']].head(10))

  YearsCodePro WorkExp
0          NaN  Expert
1           17  Expert
2           27  Expert
3          NaN  Expert
4          NaN  Expert
5          NaN  Expert
6            7  Senior
7          NaN  Expert
8          NaN  Expert
9           11  Expert

### Summary


This lab:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
